### Fire Incidents

In [193]:
import pandas as pd

path = "...FDNY\\"

df = pd.read_csv(path + "Fire_Incident_Dispatch_Data_2025.csv", low_memory=False)


In [194]:
pd.set_option('chained_assignment', None)

df["COMMUNITYDISTRICT"] = df["COMMUNITYDISTRICT"].astype(str)
#df["COMMUNITYDISTRICT"] = df["COMMUNITYDISTRICT"].astype(int)
#df["COMMUNITYDISTRICT"] = df["COMMUNITYDISTRICT"].astype(str)
print (len(df))
df = df[df["COMMUNITYDISTRICT"] != 'nan'].reset_index(drop=True)
print (len(df))

df["COMMUNITYDISTRICT"] = df["COMMUNITYDISTRICT"].str.split(".").str[0]


cd = sorted(list(set(df["COMMUNITYDISTRICT"].tolist() )))
print (len(cd))
#cd

211774
199744
71


In [195]:
df["INCIDENT_CLASSIFICATION"] = df["INCIDENT_CLASSIFICATION"].astype(str)

ic = (list(set(df["INCIDENT_CLASSIFICATION"].tolist() )))

#for c in ic:
#    print (c)

#df["INCIDENT_CLASSIFICATION2"] = df["INCIDENT_CLASSIFICATION"].str.split(" - ").str[0].reset_index(drop=True)

df["INCIDENT_CLASSIFICATION2"] = (
    df["INCIDENT_CLASSIFICATION"]
    .str.split(r"\s-\s", regex=True)
    .str[0]
    .str.strip()
)

#df["INCIDENT_CLASSIFICATION2"] = df["INCIDENT_CLASSIFICATION2"].astype(str)

ic = (list(set(df["INCIDENT_CLASSIFICATION2"].tolist() )))



In [196]:
df2 = df[["COMMUNITYDISTRICT", "INCIDENT_CLASSIFICATION2", "INCIDENT_RESPONSE_SECONDS_QY", "INCIDENT_TRAVEL_TM_SECONDS_QY" ]]

df2["COMMUNITYDISTRICT"] = df2["COMMUNITYDISTRICT"].astype(str)


dummies = pd.get_dummies(df2["INCIDENT_CLASSIFICATION2"], prefix="IC").astype(int)
df2 = pd.concat([df2, dummies], axis=1)

df2 = df2.rename(columns={"IC_Multiple Dwelling 'A'": "IC_Multiple Dwelling A", })






Create Dataframe to Merge with Shapefile

In [197]:
subset = ['COMMUNITYDISTRICT', 'INCIDENT_RESPONSE_SECONDS_QY', 'INCIDENT_TRAVEL_TM_SECONDS_QY', 'IC_Medical', 'IC_Assist Civilian', 'IC_Utility Emergency', 'IC_Alarm System', 'IC_Elevator Emergency',
         'IC_Odor', 'IC_Non-Medical MFA', 'IC_Multiple Dwelling A', 'IC_Carbon Monoxide', 'IC_Vehicle Accident', 'IC_Private Dwelling Fire']
df3 = df2[subset]




Get centroid of the community district

In [198]:
import geopandas as gpd

pathshape = "C:\\Users\\MehriD01\\OneDrive - New York City Housing Authority\\Documents\\FDNY\\nycd_26a\\"

# Load shapefile
gdf = gpd.read_file(pathshape + "nycd.shp")

# project to a projected CRS
gdf_proj = gdf.to_crs(epsg=2263)

# calculate centroid there
gdf_proj["centroid"] = gdf_proj.geometry.centroid

# convert centroid back to lat/lon
centroids = gdf_proj.set_geometry("centroid").to_crs(epsg=4326)

gdf["lon"] = centroids.geometry.x
gdf["lat"] = centroids.geometry.y

gdf = gdf[["BoroCD", "lon", "lat"]]

gdf["BoroCD"] = gdf["BoroCD"].astype(str)


Groupby

In [199]:
df4 = df3[['COMMUNITYDISTRICT', 'IC_Medical', 'IC_Assist Civilian', 'IC_Utility Emergency', 'IC_Alarm System', 'IC_Elevator Emergency',
         'IC_Odor', 'IC_Non-Medical MFA', 'IC_Multiple Dwelling A', 'IC_Carbon Monoxide', 'IC_Vehicle Accident', 'IC_Private Dwelling Fire'] ]

df4["COMMUNITYDISTRICT"] = df4["COMMUNITYDISTRICT"].astype(str)

df4G = df4.groupby(['COMMUNITYDISTRICT']).sum()
df4G = df4G.add_suffix('').reset_index()

df4G

,COMMUNITYDISTRICT,IC_Medical,IC_Assist Civilian,IC_Utility Emergency,IC_Alarm System,IC_Elevator Emergency,IC_Odor,IC_Non-Medical MFA,IC_Multiple Dwelling A,IC_Carbon Monoxide,IC_Vehicle Accident,IC_Private Dwelling Fire
0,101,933,122,93,402,166,68,67,23,21,8,2
1,102,1413,175,173,481,119,155,65,52,41,15,4
2,103,2879,414,515,337,512,211,94,129,112,17,1
3,104,2478,300,222,502,276,138,135,71,54,36,1
4,105,2917,293,209,1140,314,125,191,28,29,22,1
...,...,...,...,...,...,...,...,...,...,...,...,...
66,484,4,0,0,0,0,0,0,0,0,0,0
67,501,2446,454,294,170,160,118,68,44,79,110,121
68,502,1599,269,170,173,32,74,24,16,60,110,88
69,503,1361,281,150,73,8,86,23,9,68,72,117


Merge lat/lon and incident response times

In [200]:
gdfDic = gdf.set_index('BoroCD')['lat'].to_dict()
df4G["lat"] = df4G["COMMUNITYDISTRICT"].map(gdfDic)

gdfDic = gdf.set_index('BoroCD')['lon'].to_dict()
df4G["lon"] = df4G["COMMUNITYDISTRICT"].map(gdfDic)



Add Population by Community District

In [201]:
path

'C:\\Users\\MehriD01\\OneDrive - New York City Housing Authority\\Documents\\FDNY\\'

In [202]:
dp = pd.read_csv(path + "New_York_City_Population_By_Community_Districts_20260414.csv")
dp["CD Number"] = dp["CD Number"].astype(str)
dp["CD Number"] = dp["CD Number"].str.split(".").str[0]
dp = dp[dp["CD Number"] != 'nan']

dp['Borough2'] = dp['Borough']

dp['Borough2'] = dp['Borough2'].str.replace('Bronx', '2')
dp['Borough2'] = dp['Borough2'].str.replace('Manhattan', '1')
dp['Borough2'] = dp['Borough2'].str.replace('Brooklyn', '3')
dp['Borough2'] = dp['Borough2'].str.replace('Queens', '4')
dp['Borough2'] = dp['Borough2'].str.replace('Staten Island', '5')

dp['boro_cd'] = dp['Borough2'].astype(str) + dp['CD Number'].astype(str).str.zfill(2)

dp['2010 Population'] = dp['2010 Population'].str.replace(',', '')

dp['2010 Population'] = dp['2010 Population'].astype(int)





In [203]:
dp

,Borough,CD Number,CD Name,1970 Population,1980 Population,1990 Population,2000 Population,2010 Population,Borough2,boro_cd
0,Bronx,1,"Melrose, Mott Haven, Port Morris","138,557","78,441","77,214","82,159",91497,2,201
1,Bronx,2,"Hunts Point, Longwood","99,493","34,399","39,443","46,824",52246,2,202
2,Bronx,3,"Morrisania, Crotona Park East","150,636","53,635","57,162","68,574",79762,2,203
3,Bronx,4,"Highbridge, Concourse Village","144,207","114,312","119,962","139,563",146441,2,204
4,Bronx,5,"University Hts., Fordham, Mt. Hope","121,807","107,995","118,435","128,313",128200,2,205
5,Bronx,6,"East Tremont, Belmont","114,137","65,016","68,061","75,688",83268,2,206
6,Bronx,7,"Bedford Park, Norwood, Fordham","113,764","116,827","128,588","141,411",139286,2,207
7,Bronx,8,"Riverdale, Kingsbridge, Marble Hill","103,543","98,275","97,030","101,332",101731,2,208
8,Bronx,9,"Soundview, Parkchester","166,442","167,627","155,970","167,859",172298,2,209
9,Bronx,10,"Throgs Nk., Co-op City, Pelham Bay","84,948","106,516","108,093","115,948",120392,2,210


Merge 

In [205]:
dpDic = dp.set_index('boro_cd')['2010 Population'].to_dict()
df4G["2010 Population"] = df4G["COMMUNITYDISTRICT"].map(dpDic)

dpDic = dp.set_index('boro_cd')['CD Name'].to_dict()
df4G["Community District Name"] = df4G["COMMUNITYDISTRICT"].map(dpDic)

dpDic = dp.set_index('boro_cd')['Borough'].to_dict()
df4G["Borough"] = df4G["COMMUNITYDISTRICT"].map(dpDic)

print (len(df4G))
df4G["2010 Population"] = df4G["2010 Population"].astype(str)
df4G = df4G[df4G["2010 Population"] != 'nan'].reset_index(drop=True)
df4G["2010 Population"] = df4G["2010 Population"].astype(int)
print (len(df4G))


df4G["Medical Norm"] = (df4G["IC_Medical"]/df4G["2010 Population"] ) * 100000
df4G["Assist Civilian Norm"] = (df4G["IC_Assist Civilian"]/df4G["2010 Population"] ) * 100000
df4G["Utility Emergency Norm"] = (df4G["IC_Utility Emergency"]/df4G["2010 Population"] ) * 100000
df4G["Private Dwelling Fire Norm"] = (df4G["IC_Private Dwelling Fire"]/df4G["2010 Population"] ) * 100000

df4G["Medical Norm"] = df4G["Medical Norm"].round(0)
df4G["Assist Civilian Norm"] = df4G["Assist Civilian Norm"].round(0)
df4G["Utility Emergency Norm"] = df4G["Utility Emergency Norm"].round(0)
df4G["Private Dwelling Fire Norm"] = df4G["Private Dwelling Fire Norm"].round(0)



59
59


In [206]:
#df4G.to_csv(path + "test.csv", index=False)

Add Response Time

In [207]:
dr = df2[["COMMUNITYDISTRICT", "INCIDENT_RESPONSE_SECONDS_QY"]]

print (len(dr))

dr["INCIDENT_RESPONSE_SECONDS_QY"] = dr["INCIDENT_RESPONSE_SECONDS_QY"].astype(str)
dr = dr[dr["INCIDENT_RESPONSE_SECONDS_QY"] != 'nan']
dr["INCIDENT_RESPONSE_SECONDS_QY"] = dr["INCIDENT_RESPONSE_SECONDS_QY"].str.replace(',', '')
dr["INCIDENT_RESPONSE_SECONDS_QY"] = dr["INCIDENT_RESPONSE_SECONDS_QY"].astype(int)

print (len(dr))


drG = dr.groupby(['COMMUNITYDISTRICT']).mean().round(0)
drG = drG.add_suffix('').reset_index()

drDic = drG.set_index('COMMUNITYDISTRICT')['INCIDENT_RESPONSE_SECONDS_QY'].to_dict()
df4G["Response Time"] = df4G["COMMUNITYDISTRICT"].map(drDic)
df4G["Response Time"] = (df4G["Response Time"]/60).round(2)


dr = df2[["COMMUNITYDISTRICT",  "INCIDENT_TRAVEL_TM_SECONDS_QY"]]

dr["INCIDENT_TRAVEL_TM_SECONDS_QY"] = dr["INCIDENT_TRAVEL_TM_SECONDS_QY"].astype(str)
dr = dr[dr["INCIDENT_TRAVEL_TM_SECONDS_QY"] != 'nan']
dr["INCIDENT_TRAVEL_TM_SECONDS_QY"] = dr["INCIDENT_TRAVEL_TM_SECONDS_QY"].str.replace(',', '')
dr["INCIDENT_TRAVEL_TM_SECONDS_QY"] = dr["INCIDENT_TRAVEL_TM_SECONDS_QY"].astype(int)

print (len(dr))

drG = dr.groupby(['COMMUNITYDISTRICT']).mean().round(0)
drG = drG.add_suffix('').reset_index()

drDic = drG.set_index('COMMUNITYDISTRICT')['INCIDENT_TRAVEL_TM_SECONDS_QY'].to_dict()
df4G["Travel Time"] = df4G["COMMUNITYDISTRICT"].map(drDic)
df4G["Travel Time"] = (df4G["Travel Time"]/60).round(2)



199744
151986
151986


In [208]:
df4G

,COMMUNITYDISTRICT,IC_Medical,IC_Assist Civilian,IC_Utility Emergency,IC_Alarm System,IC_Elevator Emergency,IC_Odor,IC_Non-Medical MFA,IC_Multiple Dwelling A,IC_Carbon Monoxide,...,lon,2010 Population,Community District Name,Borough,Medical Norm,Assist Civilian Norm,Utility Emergency Norm,Private Dwelling Fire Norm,Response Time,Travel Time
0,101,933,122,93,402,166,68,67,23,21,...,-74.011997,60978,"Battery Park City, Tribeca",Manhattan,1530.0,200.0,153.0,3.0,5.77,5.20
1,102,1413,175,173,481,119,155,65,52,41,...,-74.001559,90016,"Greenwich Village, Soho",Manhattan,1570.0,194.0,192.0,4.0,5.22,4.63
2,103,2879,414,515,337,512,211,94,129,112,...,-73.985541,163277,"Lower East Side, Chinatown",Manhattan,1763.0,254.0,315.0,1.0,5.98,5.43
3,104,2478,300,222,502,276,138,135,71,54,...,-73.997537,103245,"Chelsea, Clinton",Manhattan,2400.0,291.0,215.0,1.0,6.50,5.85
4,105,2917,293,209,1140,314,125,191,28,29,...,-73.984057,51673,Midtown Business District,Manhattan,5645.0,567.0,404.0,2.0,6.47,5.82
5,106,1620,188,184,410,169,100,74,53,37,...,-73.974576,142745,"Stuyvesant Town, Turtle Bay",Manhattan,1135.0,132.0,129.0,0.0,6.02,5.42
6,107,2125,271,316,347,281,174,89,97,93,...,-73.976126,209084,"West Side, Upper West Side",Manhattan,1016.0,130.0,151.0,0.0,6.23,5.58
7,108,1778,255,269,415,198,201,61,82,70,...,-73.955718,219920,Upper East Side,Manhattan,808.0,116.0,122.0,3.0,6.63,5.88
8,109,1363,217,296,179,213,113,45,77,38,...,-73.955030,110193,"Manhattanville, Hamilton Heights",Manhattan,1237.0,197.0,269.0,0.0,6.05,5.37
9,110,2458,375,522,270,597,216,119,169,82,...,-73.944736,115723,Central Harlem,Manhattan,2124.0,324.0,451.0,0.0,6.57,5.77


Add borough to cd name

In [209]:
df4G["Community District Name"] = df4G["Community District Name"] + ", " + df4G["Borough"]

In [210]:
df4G.to_csv(path + "Fire Dispatch Data 2025.csv", index=False)

Mean and Standard Deviation

In [211]:
mean_val = df4G['Response Time'].mean()
print (mean_val)
std_val = df4G['Response Time'].std()
print (std_val)

6.089661016949153
0.6026978602048456


In [192]:
df4G.dtypes

COMMUNITYDISTRICT            object
IC_Medical                    int32
IC_Assist Civilian            int32
IC_Utility Emergency          int32
IC_Alarm System               int32
IC_Elevator Emergency         int32
IC_Odor                       int32
IC_Non-Medical MFA            int32
IC_Multiple Dwelling A        int32
IC_Carbon Monoxide            int32
IC_Vehicle Accident           int32
IC_Private Dwelling Fire      int32
lat                         float64
lon                         float64
2010 Population               int32
Community District Name      object
Borough                      object
Medical Norm                float64
Assist Civilian Norm        float64
Utility Emergency Norm      float64
Private Dwelling Fire       float64
Response Time               float64
Travel Time                 float64
dtype: object

In [22]:
cols = df2.columns
for c in cols:
    print (c)

COMMUNITYDISTRICT
INCIDENT_CLASSIFICATION2
INCIDENT_RESPONSE_SECONDS_QY
INCIDENT_TRAVEL_TM_SECONDS_QY
IC_Abandoned Derelict Vehicle Fire
IC_Alarm System
IC_Assist Civilian
IC_Automobile Fire
IC_Brush Fire
IC_Carbon Monoxide
IC_Church Fire
IC_Defective Oil Burner
IC_Demolition Debris or Rubbish Fire
IC_Downed Tree
IC_Elevator Emergency
IC_Factory Fire
IC_Hospital Fire
IC_Manhole Fire
IC_Maritime Emergency
IC_Maritime Fire
IC_Medical
IC_Medical MFA
IC_Multiple Dwelling A
IC_Multiple Dwelling 'B' Fire
IC_Non-Medical 10-91 (Unnecessary Alarm)
IC_Non-Medical MFA
IC_Odor
IC_Other Commercial Building Fire
IC_Other Public Building Fire
IC_Other Transportation Fire
IC_Private Dwelling Fire
IC_Remove Civilian
IC_School Fire
IC_Sprinkler System
IC_Store Fire
IC_Theater or TV Studio Fire
IC_Transit System
IC_Transit System Emergency
IC_Undefined Emergency
IC_Undefined Nonstructural Fire
IC_Under Contruction / Vacant Fire
IC_Utility Emergency
IC_Vehicle Accident


In [23]:
df["INCIDENT_CLASSIFICATION2"].value_counts()

INCIDENT_CLASSIFICATION2
Medical                                  108948
Assist Civilian                           15846
Utility Emergency                         15012
Alarm System                              14702
Elevator Emergency                        10683
Odor                                       6861
Non-Medical MFA                            4349
Multiple Dwelling 'A'                      4325
Carbon Monoxide                            3640
Vehicle Accident                           3509
Manhole Fire                               2326
Private Dwelling Fire                      2003
Demolition Debris or Rubbish Fire          1947
Other Commercial Building Fire             1341
Medical MFA                                 677
Sprinkler System                            648
Defective Oil Burner                        516
Downed Tree                                 459
Multiple Dwelling 'B' Fire                  451
Automobile Fire                             365
Brush Fire     

In [21]:
df.dtypes

STARFIRE_INCIDENT_ID               object
INCIDENT_DATETIME                  object
ALARM_BOX_BOROUGH                  object
ALARM_BOX_NUMBER                    int64
ALARM_BOX_LOCATION                 object
INCIDENT_BOROUGH                   object
ZIPCODE                           float64
POLICEPRECINCT                    float64
CITYCOUNCILDISTRICT               float64
COMMUNITYDISTRICT                  object
COMMUNITYSCHOOLDISTRICT           float64
CONGRESSIONALDISTRICT             float64
ALARM_SOURCE_DESCRIPTION_TX        object
ALARM_LEVEL_INDEX_DESCRIPTION      object
HIGHEST_ALARM_LEVEL                object
INCIDENT_CLASSIFICATION            object
INCIDENT_CLASSIFICATION_GROUP      object
DISPATCH_RESPONSE_SECONDS_QY       object
FIRST_ASSIGNMENT_DATETIME          object
FIRST_ACTIVATION_DATETIME          object
FIRST_ON_SCENE_DATETIME            object
INCIDENT_CLOSE_DATETIME            object
VALID_DISPATCH_RSPNS_TIME_INDC     object
VALID_INCIDENT_RSPNS_TIME_INDC    

In [9]:
df["INCIDENT_CLASSIFICATION"]

0                  Assist Civilian - Non-Medical
1                       Medical - EMS Link 10-91
2                       Medical - EMS Link 10-91
3         Medical - No PT Contact EMS is Onscene
4         Medical - No PT Contact EMS is Onscene
                           ...                  
199739                  Medical - EMS Link 10-91
199740    Medical - No PT Contact EMS is Onscene
199741                  Medical - EMS Link 10-91
199742                  Medical - EMS Link 10-91
199743             Elevator Emergency - Occupied
Name: INCIDENT_CLASSIFICATION, Length: 199744, dtype: object

In [45]:
icg = sorted(list(set(df["INCIDENT_CLASSIFICATION_GROUP"].tolist() )))
icg

['Medical Emergencies',
 'Medical MFAs',
 'NonMedical Emergencies',
 'NonMedical MFAs',
 'NonStructural Fires',
 'Structural Fires']

In [89]:
df.dtypes

STARFIRE_INCIDENT_ID               object
INCIDENT_DATETIME                  object
ALARM_BOX_BOROUGH                  object
ALARM_BOX_NUMBER                    int64
ALARM_BOX_LOCATION                 object
INCIDENT_BOROUGH                   object
ZIPCODE                           float64
POLICEPRECINCT                    float64
CITYCOUNCILDISTRICT               float64
COMMUNITYDISTRICT                  object
COMMUNITYSCHOOLDISTRICT           float64
CONGRESSIONALDISTRICT             float64
ALARM_SOURCE_DESCRIPTION_TX        object
ALARM_LEVEL_INDEX_DESCRIPTION      object
HIGHEST_ALARM_LEVEL                object
INCIDENT_CLASSIFICATION            object
INCIDENT_CLASSIFICATION_GROUP      object
DISPATCH_RESPONSE_SECONDS_QY       object
FIRST_ASSIGNMENT_DATETIME          object
FIRST_ACTIVATION_DATETIME          object
FIRST_ON_SCENE_DATETIME            object
INCIDENT_CLOSE_DATETIME            object
VALID_DISPATCH_RSPNS_TIME_INDC     object
VALID_INCIDENT_RSPNS_TIME_INDC    